# Batch Runs: bb144 + Scheduler_Test

This notebook runs batch experiments inline using both `naive_dag` and `naive_n_dag`.


In [ ]:
from pathlib import Path
import json
import re
import time
import sys

import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from naive_dag.main import load_config as load_naive_dag_config
from naive_dag.main import main as naive_dag_main
from naive_n_dag.main import load_config as load_naive_n_dag_config
from naive_n_dag.main import main as naive_n_dag_main

outputs_dir = repo_root / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

bb144_qasm = repo_root / "inputs" / "qasm_files" / "bb144_bivariate_bicycle_cz.qasm"
assert bb144_qasm.exists(), f"Missing QASM file: {bb144_qasm}"


def timed_run(fn, *args, **kwargs):
    t0 = time.perf_counter()
    exit_code = fn(*args, **kwargs)
    dt = time.perf_counter() - t0
    return exit_code, dt


def run_naive_dag_with_named_output(config_path: Path, qasm_path: Path, out_dir: Path, output_name: str, seed: int = 0):
    out_dir.mkdir(parents=True, exist_ok=True)
    config = load_naive_dag_config(config_path)
    exit_code, dt = timed_run(
        naive_dag_main,
        config,
        str(qasm_path),
        schedule_output_dir=out_dir,
        config_path=config_path,
        quiet=True,
        seed=seed,
    )
    produced = out_dir / f"{qasm_path.stem}.schedule.txt"
    target = out_dir / output_name
    if produced.exists():
        if target.exists():
            target.unlink()
        produced.rename(target)
    return exit_code, dt, target


def run_naive_n_dag_with_named_output(config_path: Path, qasm_path: Path, out_dir: Path, output_name: str, seed: int = 0, log: bool = False):
    out_dir.mkdir(parents=True, exist_ok=True)
    config = load_naive_n_dag_config(config_path)
    exit_code, dt = timed_run(
        naive_n_dag_main,
        config,
        str(qasm_path),
        schedule_output_dir=out_dir,
        config_path=config_path,
        quiet=True,
        seed=seed,
        output_name=output_name,
        log=log,
    )
    target = out_dir / output_name
    return exit_code, dt, target


def resolve_qasm_for_config(config_path: Path) -> Path:
    cfg = json.loads(config_path.read_text())

    if config_path.name.startswith("bb144_n_"):
        return bb144_qasm

    base_key = "qasm_base_dir" if "qasm_base_dir" in cfg else "qasm_dir"
    base_dir = (config_path.parent / cfg[base_key]).resolve()
    stem = config_path.stem

    candidates = [
        base_dir / f"{stem}.qasm",
        base_dir / f"{stem}_transpiled.qasm",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    hits = sorted(base_dir.glob(f"{stem}*.qasm"))
    if hits:
        return hits[0]

    raise FileNotFoundError(f"Could not resolve qasm for {config_path} from base_dir={base_dir}")


## 1) Run bb144 k-sweep (`k1` to `k100`) inline

This runs all available `inputs/algorithms/bb144_n_fastsa_k*.json` with `k <= 100` using both solvers.
Outputs are written under `outputs/bb144_k_sweep/`.


In [ ]:
algo_dir = repo_root / "inputs" / "algorithms"
k_sweep_out_root = outputs_dir / "bb144_k_sweep"

pattern = re.compile(r"bb144_n_fastsa_k(\d+)\.json$")
configs = []
for path in sorted(algo_dir.glob("bb144_n_fastsa_k*.json")):
    m = pattern.match(path.name)
    if not m:
        continue
    k = int(m.group(1))
    if 1 <= k <= 100:
        configs.append((k, path))

configs.sort(key=lambda x: x[0])
print("k-sweep configs:", [p.name for _, p in configs])

records = []
for k, cfg_path in configs:
    n_out_name = f"{cfg_path.stem}.naive_n_dag.schedule.txt"
    d_out_name = f"{cfg_path.stem}.naive_dag.schedule.txt"

    n_exit, n_sec, n_path = run_naive_n_dag_with_named_output(
        cfg_path,
        bb144_qasm,
        k_sweep_out_root / "naive_n_dag",
        n_out_name,
        seed=0,
        log=False,
    )
    records.append(
        {
            "config": cfg_path.name,
            "k": k,
            "solver": "naive_n_dag",
            "wall_seconds": n_sec,
            "exit_code": n_exit,
            "output_file": str(n_path.relative_to(repo_root)),
        }
    )

    d_exit, d_sec, d_path = run_naive_dag_with_named_output(
        cfg_path,
        bb144_qasm,
        k_sweep_out_root / "naive_dag",
        d_out_name,
        seed=0,
    )
    records.append(
        {
            "config": cfg_path.name,
            "k": k,
            "solver": "naive_dag",
            "wall_seconds": d_sec,
            "exit_code": d_exit,
            "output_file": str(d_path.relative_to(repo_root)),
        }
    )

k_sweep_df = pd.DataFrame(records).sort_values(["solver", "k"]).reset_index(drop=True)
k_sweep_df


`k` is the number of greedy iterations. The next block parses runtime and plots execution time vs `k`.


In [ ]:
plot_df = (
    k_sweep_df[k_sweep_df["solver"] == "naive_n_dag"]
    .sort_values("k")
    .reset_index(drop=True)
)

if plot_df.empty:
    raise RuntimeError("No k-sweep records for naive_n_dag to plot.")

plt.figure(figsize=(8, 4))
plt.plot(plot_df["k"], plot_df["wall_seconds"], marker="o", linewidth=2)
plt.xlabel("k (number of greedy iterations)")
plt.ylabel("Execution time (wall seconds)")
plt.title("bb144 k-sweep: naive_n_dag execution time vs k")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plot_df[["k", "wall_seconds", "exit_code", "output_file"]]


## 2) Batch run: `bb144_n_fastsa_k1000.json` (`--log`), `bb144_n_init.json`, and all `Scheduler_Test/INPUT/*.json`

This block runs both `naive_dag` and `naive_n_dag` inline and writes outputs to `Scheduler_Test/OUTPUT/`.


In [ ]:
scheduler_output_dir = repo_root / "Scheduler_Test" / "OUTPUT"
scheduler_output_dir.mkdir(parents=True, exist_ok=True)

batch_configs = [
    repo_root / "inputs" / "algorithms" / "bb144_n_fastsa_k1000.json",
    repo_root / "inputs" / "algorithms" / "bb144_n_init.json",
]
batch_configs.extend(sorted((repo_root / "Scheduler_Test" / "INPUT").glob("*.json")))

batch_records = []
for cfg_path in batch_configs:
    qasm_path = resolve_qasm_for_config(cfg_path)

    run_log = cfg_path.name == "bb144_n_fastsa_k1000.json"

    n_output_name = f"{cfg_path.stem}.naive_n_dag.schedule.txt"
    n_exit, n_sec, n_path = run_naive_n_dag_with_named_output(
        cfg_path,
        qasm_path,
        scheduler_output_dir,
        n_output_name,
        seed=0,
        log=run_log,
    )
    batch_records.append(
        {
            "config": str(cfg_path.relative_to(repo_root)),
            "solver": "naive_n_dag",
            "qasm": str(qasm_path.relative_to(repo_root)),
            "wall_seconds": n_sec,
            "exit_code": n_exit,
            "output_file": str(n_path.relative_to(repo_root)),
            "log_enabled": run_log,
        }
    )

    d_output_name = f"{cfg_path.stem}.naive_dag.schedule.txt"
    d_exit, d_sec, d_path = run_naive_dag_with_named_output(
        cfg_path,
        qasm_path,
        scheduler_output_dir,
        d_output_name,
        seed=0,
    )
    batch_records.append(
        {
            "config": str(cfg_path.relative_to(repo_root)),
            "solver": "naive_dag",
            "qasm": str(qasm_path.relative_to(repo_root)),
            "wall_seconds": d_sec,
            "exit_code": d_exit,
            "output_file": str(d_path.relative_to(repo_root)),
            "log_enabled": False,
        }
    )

batch_df = pd.DataFrame(batch_records).sort_values(["config", "solver"]).reset_index(drop=True)
batch_df
